# Prototipo de Anotación Clínica con LLM — TDAH


Modelo LLM local que convierte el texto libre de una observación parental en una anotación clínica estructurada.

**Características del prototipo:**
- Permite usar diferentes instrumentos clínicos (BRIEF-2, para la demo) se carga desde un fichero externo configurable.
  Cambiar de instrumento (SNAP-IV, Conners, SDQ...) = cambiar el fichero JSON
- La evaluación de la salida no requiere ground truth: mide propiedades estructurales y
  de coherencia interna de la anotación.


In [6]:
!curl http://localhost:11002/api/tags

{"models":[{"name":"gemma4:e2b","model":"gemma4:e2b","modified_at":"2026-06-13T10:24:09.203280913Z","size":7162405886,"digest":"7fbdbf8f5e45a75bb122155ed546e765b4d9c53a1285f62fd9f506baa1c5a47e","details":{"parent_model":"","format":"gguf","family":"gemma4","families":["gemma4"],"parameter_size":"5.1B","quantization_level":"Q4_K_M"},"capabilities":["completion","tools","thinking"]},{"name":"gemma4:e4b","model":"gemma4:e4b","modified_at":"2026-06-13T10:22:53.48880674Z","size":9608350718,"digest":"c6eb396dbd5992bbe3f5cdb947e8bbc0ee413d7c17e2beaae69f5d569cf982eb","details":{"parent_model":"","format":"gguf","family":"gemma4","families":["gemma4"],"parameter_size":"8.0B","quantization_level":"Q4_K_M"},"capabilities":["completion","tools","thinking"]},{"name":"gemma4:12b","model":"gemma4:12b","modified_at":"2026-06-13T10:21:20.365041293Z","size":7556508396,"digest":"4eb23ef187e2c5462566d6a1d3bbbc2f1346d0b4327cbb66d58fffbcc9b2b05c","details":{"parent_model":"","format":"gguf","family":"gemma4

---
## FASE 0 — Entorno

No se usa LangChain ni frameworks pesados para esta priemra versión del prototipo, así permite ver la llamada HTTP directa al modelo.

In [ ]:
import json
import os
import re
import requests
from pathlib import Path

# ── Configuración  ──────────────

OLLAMA_PORT = os.environ.get("OLLAMA_PORT","11002")
OLLAMA_BASE = f"http://127.0.0.1:{OLLAMA_PORT}"
OLLAMA_URL  = f"{OLLAMA_BASE}/api/generate"
TEMPERATURE = 0.1                            
GEMMA4_PREFERENCIAS = (
    "gemma4:26b",
    "gemma4:31b",
    "gemma4:12b",
    "gemma4:e4b",
    "gemma4:e2b",
    "gemma4",
)

def listar_modelos_ollama() -> list[str]:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    return [m["name"] for m in r.json().get("models", [])]

def encontrar_gemma4(modelos: list[str]) -> str | None:
    for tag in GEMMA4_PREFERENCIAS:
        coincidencias = [
            m for m in modelos
            if m == tag or m.startswith(tag + ":") or m.startswith(tag + "-")
        ]
        if not coincidencias:
            continue
        return next((m for m in coincidencias if m == tag), coincidencias[0])
    return None

def resolver_modelo() -> str:
    if os.environ.get("OLLAMA_MODEL"):
        return os.environ["OLLAMA_MODEL"]
    try:
        if encontrado := encontrar_gemma4(listar_modelos_ollama()):
            return encontrado
    except Exception:
        pass
    return "gemma4:26b"  

MODEL = resolver_modelo()

print("Configuración:")
print(f"  Modelo:      {MODEL}")
print(f"  Puerto:      {OLLAMA_PORT}")
print(f"  Endpoint:    {OLLAMA_URL}")
print(f"  Temperatura: {TEMPERATURE}")

Configuración:
  Modelo:      gemma4:26b
  Puerto:      11002
  Endpoint:    http://127.0.0.1:11002/api/generate
  Temperatura: 0.1


In [2]:
def comprobar_ollama():
    global MODEL
    try:
        modelos = listar_modelos_ollama()
        gemma4 = [m for m in modelos if m.startswith("gemma4")]
        print(f"✅ Ollama accesible. Gemma 4 locales: {gemma4 or 'ninguno'}")
        if encontrado := encontrar_gemma4(modelos):
            MODEL = encontrado
            print(f"✅ Usando modelo: {MODEL}")
        else:
            print(f"⚠️  Ningún Gemma 4 encontrado. Descargar uno (recomendado ~27B):")
            print(f"   OLLAMA_HOST=127.0.0.1:{OLLAMA_PORT} ollama pull gemma4:26b")
    except Exception as e:
        print(f"❌ No se puede conectar a Ollama en {OLLAMA_BASE}: {e}")
        print(f"   Arrancar en mercurio (puerto {OLLAMA_PORT}, no 11434):")
        print(f"   OLLAMA_HOST=127.0.0.1:{OLLAMA_PORT} ollama serve")

comprobar_ollama()

✅ Ollama accesible. Gemma 4 locales: ['gemma4:e2b', 'gemma4:e4b', 'gemma4:12b', 'gemma4:31b', 'gemma4:26b']
✅ Usando modelo: gemma4:26b


---
## FASE 1 — Carga del Cuestionario (configurable)

El cuestionario se carga como un dato de entrada. Esto convierte el sistema en un **método general de fenotipado de observaciones clínicas**, demostrado con BRIEF-2 pero aplicable a cualquier cuestionario con estructura de ítems agrupados en escalas (SNAP-IV, Conners, SDQ, Vanderbilt...).

Para anotar con otro instrumento, solo se cambia el fichero `instrumentos/brief2.json`
por otro con la misma estructura. El código de anotación no se toca.

In [4]:
def cargar_instrumento(ruta):
    """Carga un instrumento clínico desde un fichero JSON."""
    with open(ruta, encoding="utf-8") as f:
        return json.load(f)

instrumento = cargar_instrumento("instrumentos/brief2.json")

print(f"Instrumento cargado: {instrumento['nombre']}")
print(f"Descripción: {instrumento['descripcion']}")
print(f"Nº de ítems: {len(instrumento['items'])}")
print(f"Nº de escalas: {len(instrumento['escalas'])}")
print()
print("Escalas:")
for escala, desc in instrumento['escalas'].items():
    print(f"  - {escala}: {desc}")

Instrumento cargado: BRIEF-2 Familia
Descripción: Behavior Rating Inventory of Executive Function, 2ª edición. Versión Familia. Evalúa funciones ejecutivas en niños y adolescentes de 5 a 18 años.
Nº de ítems: 63
Nº de escalas: 9

Escalas:
  - inhibicion: Control de impulsos y conducta hiperactiva
  - supervision_conducta: Conciencia del impacto de la propia conducta en los demás
  - flexibilidad: Adaptación a cambios, tolerancia a la frustración, rigidez
  - control_emocional: Regulación emocional, labilidad, reactividad
  - iniciativa: Capacidad de iniciar tareas y actividades de forma autónoma
  - memoria_trabajo: Atención sostenida, retención de información en tareas
  - planificacion: Anticipación, organización del trabajo, resolución de problemas
  - supervision_tarea: Monitorización del propio trabajo, detección de errores
  - organizacion_materiales: Orden del espacio de trabajo y los materiales personales


**Visualización de los ítems.** Visualización de los primeros ítems del cuestionario. Cada ítem tiene
un número, la escala a la que pertenece y el texto literal del cuestionario.

In [5]:
print("Primeros 10 ítems del instrumento:\n")
for item in instrumento['items'][:10]:
    print(f"  {item['id']:2d} [{item['escala']:22s}] {item['texto']}")
print(f"\n  ... y {len(instrumento['items'])-10} ítems más.")

Primeros 10 ítems del instrumento:

   1 [inhibicion            ] Es inquieto o inquieta.
   2 [flexibilidad          ] Se resiste o le cuesta aceptar maneras alternativas de resolver un problema.
   3 [memoria_trabajo       ] Cuando se le pide que haga tres cosas, solo se acuerda de la primera o de la última.
   4 [supervision_conducta  ] Le cuesta darse cuenta de cómo su conducta afecta o molesta a los demás.
   5 [supervision_tarea     ] Su trabajo es descuidado.
   6 [control_emocional     ] Tiene explosiones de ira.
   7 [planificacion         ] Hace sus tareas o deberes sin planificarse previamente.
   8 [organizacion_materiales] No encuentra sus cosas en su habitación o en su mesa.
   9 [iniciativa            ] Le cuesta iniciar actividades por sí mismo o por sí misma.
  10 [inhibicion            ] Actúa sin haber pensado antes (es impulsivo o impulsiva).

  ... y 53 ítems más.


---
## FASE 2 — El texto de entrada 

**Archivo enviado por el hospital:** , tal como lo enviaría el hospital. Sin anotación previa, sin BRIEF-2, solo el texto y los datos mínimos del paciente (edad, sexo) necesarios para la interpretación clínica.

In [6]:
with open("data/mensaje_ejemplo.json", encoding="utf-8") as f:
    mensaje = json.load(f)

print("MENSAJE RECIBIDO (formato hospital):")
print(f"  Paciente:   {mensaje['id_paciente']} ({mensaje['edad']} años, {mensaje['sexo']})")
print(f"  Informante: {mensaje['rol_informante']} ({mensaje['id_familiar']})")
print(f"  Fecha:      {mensaje['fecha']}")
print(f"\n  Texto:\n  \"{mensaje['entrada']}\"")

MENSAJE RECIBIDO (formato hospital):
  Paciente:   PAC_T01 (8 años, masculino)
  Informante: madre (FAM_T01_01)
  Fecha:      2024-01-08

  Texto:
  "Primera semana con la pastilla. Marcos ha estado más tranquilo en casa, pero en el colegio la maestra dice que sigue levantándose de la silla. Le costó dormirse el lunes y el martes. Los deberes los hizo en 40 minutos, antes tardaba casi dos horas."


In [7]:
def construir_prompt_sistema(instrumento):
    """Genera el prompt de sistema a partir de la definición del instrumento."""
    # Catálogo de ítems
    catalogo = "\n".join(
        f"  {it['id']}: [{it['escala']}] {it['texto']}"
        for it in instrumento['items']
    )
    # Descripción de escalas
    escalas_desc = "\n".join(
        f"  - {e}: {d}" for e, d in instrumento['escalas'].items()
    )
    # Niveles de alerta
    niveles = " | ".join(instrumento['niveles_alerta'])

    prompt = f"""Eres un {instrumento['rol_anotador']}.

Tu tarea es analizar el texto libre de observación de un padre/madre sobre su hijo/a
y producir una anotación clínica estructurada en formato JSON, basada en el instrumento
{instrumento['nombre']}.

## CATÁLOGO DE ÍTEMS ({len(instrumento['items'])} ítems)
{catalogo}

## ESCALAS
{escalas_desc}

## NIVELES DE ALERTA
{niveles}

## INSTRUCCIONES DE SALIDA
Responde ÚNICAMENTE con un objeto JSON válido, sin texto antes ni después, sin markdown.
Estructura requerida:
{{
  "items_detectados": [lista de números de ítem observables en el texto],
  "escalas_afectadas": [lista de escalas correspondientes],
  "nivel_alerta": "{instrumento['niveles_alerta'][0]}|{instrumento['niveles_alerta'][1]}|{instrumento['niveles_alerta'][2]}",
  "nota_clinica": "resumen clínico de 1-3 frases para el médico",
  "justificacion": "explicación del razonamiento (para auditoría)"
}}"""
    return prompt

SYSTEM_PROMPT = construir_prompt_sistema(instrumento)

def mostrar_prompt_resumen(prompt: str, items_preview: int = 8) -> None:
    """Vista previa legible: cabecera, primeros ítems y secciones finales completas."""
    lineas = prompt.splitlines()
    idx_catalogo = next(i for i, l in enumerate(lineas) if l.startswith("## CATÁLOGO"))
    idx_escalas = lineas.index("## ESCALAS")

    items = [l for l in lineas[idx_catalogo + 1:idx_escalas] if l.strip()]
    omitidos = len(items) - items_preview

    print("\n".join(lineas[:idx_catalogo + 1]))
    print("\n".join(items[:items_preview]))
    if omitidos > 0:
        print(f"\n  [... {omitidos} ítems más ...]\n")
    print("\n".join(lineas[idx_escalas:]))

mostrar_prompt_resumen(SYSTEM_PROMPT)

Eres un psicólogo clínico infantil especializado en TDAH y en el instrumento BRIEF-2.

Tu tarea es analizar el texto libre de observación de un padre/madre sobre su hijo/a
y producir una anotación clínica estructurada en formato JSON, basada en el instrumento
BRIEF-2 Familia.

## CATÁLOGO DE ÍTEMS (63 ítems)
  1: [inhibicion] Es inquieto o inquieta.
  2: [flexibilidad] Se resiste o le cuesta aceptar maneras alternativas de resolver un problema.
  3: [memoria_trabajo] Cuando se le pide que haga tres cosas, solo se acuerda de la primera o de la última.
  4: [supervision_conducta] Le cuesta darse cuenta de cómo su conducta afecta o molesta a los demás.
  5: [supervision_tarea] Su trabajo es descuidado.
  6: [control_emocional] Tiene explosiones de ira.
  7: [planificacion] Hace sus tareas o deberes sin planificarse previamente.
  8: [organizacion_materiales] No encuentra sus cosas en su habitación o en su mesa.

  [... 55 ítems más ...]

## ESCALAS
  - inhibicion: Control de impulsos y co

**El prompt de usuario** combina el contexto del paciente con el texto de la observación.
Aquí es donde se inserta el mensaje concreto a anotar.

In [8]:
def construir_prompt_usuario(mensaje):
    """Genera el prompt de usuario con el contexto del paciente y su observación."""
    return f"""## CONTEXTO DEL PACIENTE
- Edad: {mensaje['edad']} años
- Sexo: {mensaje['sexo']}
- Informante: {mensaje['rol_informante']}

## TEXTO DEL PADRE/MADRE
\"\"\"{mensaje['entrada']}\"\"\"

Analiza el texto y genera el JSON de anotación clínica."""

USER_PROMPT = construir_prompt_usuario(mensaje)
print(USER_PROMPT)

## CONTEXTO DEL PACIENTE
- Edad: 8 años
- Sexo: masculino
- Informante: madre

## TEXTO DEL PADRE/MADRE
"""Primera semana con la pastilla. Marcos ha estado más tranquilo en casa, pero en el colegio la maestra dice que sigue levantándose de la silla. Le costó dormirse el lunes y el martes. Los deberes los hizo en 40 minutos, antes tardaba casi dos horas."""

Analiza el texto y genera el JSON de anotación clínica.


---
## FASE 4 — Gemma Procesa el texto en base del instrumento y del prompt



 la respuesta del modelo es texto. Aún no es JSON parseado: es lo que el
modelo ha escrito, que esperamos que sea un JSON válido pero que hay que procesar después
(Fase 5). Veremos la respuesta tal cual sale del modelo.

In [9]:
def llamar_modelo(system_prompt, user_prompt):
    """Llama a Ollama y devuelve la respuesta cruda del modelo."""
    payload = {
        "model":   MODEL,
        "system":  system_prompt,
        "prompt":  user_prompt,
        "stream":  False,
        "options": {"temperature": TEMPERATURE, "num_predict": 2048},
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=180)
    r.raise_for_status()
    return r.json()["response"]

# Ejecutar la anotación
import time
t0 = time.time()
respuesta_cruda = llamar_modelo(SYSTEM_PROMPT, USER_PROMPT)
latencia = time.time() - t0

print(f"⏱️  Latencia: {latencia:.1f} segundos\n")
print("RESPUESTA CRUDA DEL MODELO:")
print("─" * 60)
print(respuesta_cruda)

⏱️  Latencia: 13.0 segundos

RESPUESTA CRUDA DEL MODELO:
────────────────────────────────────────────────────────────
{
  "items_detectados": [30],
  "escalas_afectadas": ["inhibicion"],
  "nivel_alerta": "bajo",
  "nota_clinica": "El paciente muestra una respuesta favorable al tratamiento farmacológico con mayor tranquilidad en el hogar y optimización del tiempo en tareas escolares, aunque persiste la hiperactividad motora (levantarse de la silla) en el entorno escolar.",
  "justificacion": "Solo se identifica explícitamente el ítem 30 (levantarse de la silla). El resto de la información describe una mejoría en la eficiencia de las tareas y un síntoma no catalogado (dificultad para dormir), lo que sugiere una evolución positiva a pesar del síntoma residual."
}


---
## FASE 5 — De texto a JSON estructurado



 La función de parsing ayuda de forma robusta la salida del json.

In [10]:
def parsear_json(texto_crudo):
    """Extrae el primer JSON válido de la respuesta del modelo."""
    # Intento directo
    try:
        return json.loads(texto_crudo.strip())
    except json.JSONDecodeError:
        pass
    # Buscar bloque entre llaves
    m = re.search(r'\{.*\}', texto_crudo, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    # Quitar markdown
    limpio = re.sub(r'```(?:json)?', '', texto_crudo).strip()
    try:
        return json.loads(limpio)
    except json.JSONDecodeError:
        return None

anotacion = parsear_json(respuesta_cruda)

if anotacion:
    print("✅ JSON parseado correctamente\n")
    print(json.dumps(anotacion, ensure_ascii=False, indent=2))
else:
    print("❌ No se pudo parsear el JSON. Revisar la respuesta cruda de la Fase 4.")

✅ JSON parseado correctamente

{
  "items_detectados": [
    30
  ],
  "escalas_afectadas": [
    "inhibicion"
  ],
  "nivel_alerta": "bajo",
  "nota_clinica": "El paciente muestra una respuesta favorable al tratamiento farmacológico con mayor tranquilidad en el hogar y optimización del tiempo en tareas escolares, aunque persiste la hiperactividad motora (levantarse de la silla) en el entorno escolar.",
  "justificacion": "Solo se identifica explícitamente el ítem 30 (levantarse de la silla). El resto de la información describe una mejoría en la eficiencia de las tareas y un síntoma no catalogado (dificultad para dormir), lo que sugiere una evolución positiva a pesar del síntoma residual."
}


---
## FASE 6 — Evaluación de la anotación (sin ground truth)

 *¿cómo medimos si el sistema funciona bien si todavía no tenemos anotaciones de clínicos?*


| Métrica | Qué verifica | ¿Necesita ground truth? |
|---------|-------------|------------------------|
| Validez de ítems | Todos los ítems están en el rango del instrumento | No |
| Validez de escalas | Las escalas existen en el instrumento | No |
| Coherencia ítem-escala | Cada escala declarada tiene un ítem que la respalda | No |
| Coherencia alerta-ítems | El nivel de alerta es proporcional al nº de ítems | No |
| Longitud de nota | La nota clínica tiene una longitud razonable | No |


In [11]:
def evaluar_anotacion(anotacion, instrumento):
    """Evalúa la anotación con métricas que no requieren ground truth."""
    resultados = {}
    reglas = instrumento['reglas_coherencia']

    # Mapa ítem -> escala desde el instrumento
    item_escala = {it['id']: it['escala'] for it in instrumento['items']}
    n_items_instrumento = len(instrumento['items'])
    escalas_validas = set(instrumento['escalas'].keys())

    items   = anotacion.get('items_detectados', [])
    escalas = anotacion.get('escalas_afectadas', [])
    nivel   = anotacion.get('nivel_alerta', '')
    nota    = anotacion.get('nota_clinica', '')

    # 1. Validez de ítems
    items_invalidos = [i for i in items if i not in item_escala]
    resultados['items_validos'] = {
        'pasa': len(items_invalidos) == 0,
        'detalle': f"{len(items)} ítems, {len(items_invalidos)} inválidos: {items_invalidos or 'ninguno'}"
    }

    # 2. Validez de escalas
    escalas_invalidas = [e for e in escalas if e not in escalas_validas]
    resultados['escalas_validas'] = {
        'pasa': len(escalas_invalidas) == 0,
        'detalle': f"{escalas_invalidas or 'todas válidas'}"
    }

    # 3. Coherencia ítem-escala
    escalas_de_items = {item_escala[i] for i in items if i in item_escala}
    huerfanas = set(escalas) - escalas_de_items
    resultados['coherencia_item_escala'] = {
        'pasa': len(huerfanas) == 0,
        'detalle': f"escalas sin ítem que las respalde: {huerfanas or 'ninguna'}"
    }

    # 4. Coherencia alerta-ítems
    min_alto = reglas['alerta_alto_min_items']
    min_mod  = reglas['alerta_moderado_min_items']
    if nivel == 'alto':
        pasa = len(items) >= min_alto
    elif nivel == 'moderado':
        pasa = len(items) >= min_mod
    else:
        pasa = True
    resultados['coherencia_alerta'] = {
        'pasa': pasa,
        'detalle': f"nivel '{nivel}' con {len(items)} ítems"
    }

    # 5. Longitud de nota clínica
    n_palabras = len(nota.split())
    pasa_long = reglas['nota_clinica_min_palabras'] <= n_palabras <= reglas['nota_clinica_max_palabras']
    resultados['longitud_nota'] = {
        'pasa': pasa_long,
        'detalle': f"{n_palabras} palabras (rango {reglas['nota_clinica_min_palabras']}-{reglas['nota_clinica_max_palabras']})"
    }

    return resultados

evaluacion = evaluar_anotacion(anotacion, instrumento)

print("EVALUACIÓN TÉCNICA (sin ground truth):\n")
n_pasa = 0
for metrica, res in evaluacion.items():
    icono = "✅" if res['pasa'] else "❌"
    print(f"  {icono} {metrica:24s} → {res['detalle']}")
    if res['pasa']: n_pasa += 1

print(f"\n  Resultado: {n_pasa}/{len(evaluacion)} métricas superadas")

EVALUACIÓN TÉCNICA (sin ground truth):

  ✅ items_validos            → 1 ítems, 0 inválidos: ninguno
  ✅ escalas_validas          → todas válidas
  ✅ coherencia_item_escala   → escalas sin ítem que las respalde: ninguna
  ✅ coherencia_alerta        → nivel 'bajo' con 1 ítems
  ✅ longitud_nota            → 35 palabras (rango 10-120)

  Resultado: 5/5 métricas superadas


---
## FASE 7 — Experimentar y comparar

Experimentos:

- **Cambiar el texto de entrada**: probar con otra observación parental.
- **Cambiar el prompt**: editar `construir_prompt_sistema` y ver si mejora la anotación.
- **Cambiar el instrumento**: cargar otro fichero JSON con un cuestionario distinto.
- **Cambiar la temperatura**: subir a 0.3 o 0.5 y observar si la salida varía.

La función de abajo encapsula todo el pipeline (Fases 3 a 6) en una sola llamada, para
poder iterar rápido.

In [12]:
def anotar_completo(mensaje, instrumento, verbose=True):
    """Pipeline completo: prompt -> modelo -> parsing -> evaluación."""
    system = construir_prompt_sistema(instrumento)
    user   = construir_prompt_usuario(mensaje)

    t0 = time.time()
    cruda = llamar_modelo(system, user)
    latencia = time.time() - t0

    anotacion = parsear_json(cruda)
    if anotacion is None:
        print("❌ Error de parsing"); return None

    evaluacion = evaluar_anotacion(anotacion, instrumento)
    n_pasa = sum(1 for r in evaluacion.values() if r['pasa'])

    if verbose:
        nivel = anotacion.get('nivel_alerta','?')
        print(f"Anotación de {mensaje['id_paciente']} ({latencia:.1f}s):")
        print(f"  Ítems detectados: {anotacion.get('items_detectados', [])}")
        print(f"  Escalas:          {anotacion.get('escalas_afectadas', [])}")
        print(f"  Nivel alerta:     {nivel}")
        print(f"  Nota clínica:     {anotacion.get('nota_clinica','')}")
        print(f"  Evaluación:       {n_pasa}/{len(evaluacion)} métricas OK")

    return {"anotacion": anotacion, "evaluacion": evaluacion, "latencia": latencia}

# Ejemplo: anotar el mismo mensaje de nuevo
resultado = anotar_completo(mensaje, instrumento)

Anotación de PAC_T01 (6.5s):
  Ítems detectados: [30]
  Escalas:          ['inhibicion']
  Nivel alerta:     bajo
  Nota clínica:     El paciente muestra una respuesta favorable al tratamiento farmacológico con mayor tranquilidad en el hogar y reducción del tiempo de ejecución de tareas, aunque persiste la inquietud motora (levantarse de la silla) en el entorno escolar.
  Evaluación:       5/5 métricas OK


**Ejemplo de experimento: probar con otro texto.** Modifique el texto de la variable
`nuevo_mensaje` y ejecute para anotar una observación diferente.

In [13]:
nuevo_mensaje = {
    "id_paciente": "PAC_T02",
    "id_familiar": "FAM_T02_01",
    "rol_informante": "madre",
    "fecha": "2024-01-08",
    "edad": 11,
    "sexo": "femenino",
    # ← MODIFICAR ESTE TEXTO PARA EXPERIMENTAR
    "entrada": "Sofía ha tenido una semana difícil. Lloró mucho el jueves porque pensó "
               "que una amiga estaba enfadada con ella. En clase la profesora dice que se "
               "distrae y se pierde en los detalles. El apetito sigue muy bajo con la medicación."
}

resultado2 = anotar_completo(nuevo_mensaje, instrumento)

Anotación de PAC_T02 (7.4s):
  Ítems detectados: [15, 27]
  Escalas:          ['memoria_trabajo', 'control_emocional']
  Nivel alerta:     moderado
  Nota clínica:     La paciente presenta episodios de reactividad emocional intensa ante situaciones sociales y dificultades en la atención sostenida, caracterizadas por una tendencia a perderse en detalles. Se observa un impacto significativo en su estabilidad emocional durante la última semana.
  Evaluación:       5/5 métricas OK


---
## FASE 8 — Comparación con Tool Calling Klusty

Hasta ahora hemos usado el enfoque **"pedir JSON en el prompt y parsear el texto"**.


**La diferencia conceptual:**

| | Prompt + parse JSON (Fases 3-5) | Tool calling (Klusty) |
|---|---|---|
| Cómo se estructura | Le pedimos al modelo que escriba JSON | Definimos una *función* con su esquema; el modelo rellena los argumentos |
| Garantía de formato | Débil: el modelo puede escribir texto mal formado → necesitamos `parsear_json` | Fuerte: el runtime fuerza que la salida cumpla el esquema |
| Validación de tipos | Posterior (métricas T2, T3 comprueban rangos *después*) | Integrada (enum, min/max, required se aplican *durante* la generación) |



### 8.1 Definir el instrumento como un *tool* (esquema OpenAI-compatible)

El enfoque de Klusty define la salida como una función con campos tipados. Igual que en
la Fase 1 el instrumento se cargaba de un fichero, aquí lo traducimos a un esquema de
herramienta. Los campos llevan restricciones (enum para nivel_alerta, items como array
de enteros) que el modelo está obligado a respetar.

In [14]:
def construir_tool_schema(instrumento):
    """Traduce el instrumento a un esquema de tool OpenAI-compatible.

    A diferencia del prompt de la Fase 3 (texto libre que pide JSON), aquí
    definimos un contrato formal: tipos, enums y campos obligatorios.
    """
    ids_validos = [it["id"] for it in instrumento["items"]]
    escalas_validas = list(instrumento["escalas"].keys())

    return {
        "type": "function",
        "function": {
            "name": "registrar_anotacion_clinica",
            "description": (
                f"Registra la anotación clínica estructurada de una observación "
                f"parental según el instrumento {instrumento['nombre']}."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "items_detectados": {
                        "type": "array",
                        "items": {"type": "integer", "minimum": min(ids_validos),
                                  "maximum": max(ids_validos)},
                        "description": "Números de ítem del instrumento observables en el texto.",
                    },
                    "escalas_afectadas": {
                        "type": "array",
                        "items": {"type": "string", "enum": escalas_validas},
                        "description": "Escalas correspondientes a los ítems detectados.",
                    },
                    "nivel_alerta": {
                        "type": "string",
                        "enum": instrumento["niveles_alerta"],
                        "description": "Nivel de alerta clínica de la semana.",
                    },
                    "nota_clinica": {
                        "type": "string",
                        "description": "Resumen clínico de 1-3 frases para el médico.",
                    },
                    "justificacion": {
                        "type": "string",
                        "description": "Explicación del razonamiento (auditoría).",
                    },
                },
                "required": ["items_detectados", "escalas_afectadas",
                             "nivel_alerta", "nota_clinica"],
            },
        },
    }

tool_schema = construir_tool_schema(instrumento)
print("Esquema de tool generado desde el instrumento:\n")
print(json.dumps(tool_schema, ensure_ascii=False, indent=2)[:1100])
print("\n  [...]")

Esquema de tool generado desde el instrumento:

{
  "type": "function",
  "function": {
    "name": "registrar_anotacion_clinica",
    "description": "Registra la anotación clínica estructurada de una observación parental según el instrumento BRIEF-2 Familia.",
    "parameters": {
      "type": "object",
      "properties": {
        "items_detectados": {
          "type": "array",
          "items": {
            "type": "integer",
            "minimum": 1,
            "maximum": 63
          },
          "description": "Números de ítem del instrumento observables en el texto."
        },
        "escalas_afectadas": {
          "type": "array",
          "items": {
            "type": "string",
            "enum": [
              "inhibicion",
              "supervision_conducta",
              "flexibilidad",
              "control_emocional",
              "iniciativa",
              "memoria_trabajo",
              "planificacion",
              "supervision_tarea",
              

### 8.2 Llamar al modelo con tool calling

Ollama soporta tool calling vía el endpoint `/api/chat` (no `/api/generate`). El modelo,
en lugar de escribir texto libre, devuelve una llamada a la función con los argumentos ya
estructurados. No hace falta parsear: los argumentos vienen como objeto.

**Nota:** el soporte de tool calling depende del modelo. Gemma 4 lo soporta de forma
parcial; modelos como Qwen 2.5 (el usado por Klusty es QwQ-32B) o Llama 3.1 lo soportan
de forma más completa.

In [ ]:
def llamar_modelo_tool(instrumento, mensaje, tool_schema):
    """Llama a Ollama usando tool calling (endpoint /api/chat)."""
    system = (f"Eres un {instrumento['rol_anotador']}. Analiza la observación "
              f"parental y registra la anotación llamando a la función proporcionada.")
    user = construir_prompt_usuario(mensaje)

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        "tools": [tool_schema],
        "stream": False,
        "options": {"temperature": TEMPERATURE},
    }
    r = requests.post(f"{OLLAMA_BASE}/api/chat", json=payload, timeout=180)
    r.raise_for_status()
    resp = r.json()

    # Extraer la llamada a la función
    msg = resp.get("message", {})
    tool_calls = msg.get("tool_calls", [])
    if tool_calls:
        args = tool_calls[0]["function"]["arguments"]
        
        if isinstance(args, str):
            args = json.loads(args)
        return args, "tool_calling"
  
    return parsear_json(msg.get("content", "")), "fallback_texto"

import time
print("Ejecutando anotación con TOOL CALLING...\n")
t0 = time.time()
try:
    anotacion_tool, metodo = llamar_modelo_tool(instrumento, mensaje, tool_schema)
    lat_tool = time.time() - t0
    print(f"⏱️  Latencia: {lat_tool:.1f}s | Método: {metodo}\n")
    if anotacion_tool:
        print(json.dumps(anotacion_tool, ensure_ascii=False, indent=2))
    else:
        print("⚠️  El modelo no devolvió una llamada de función válida.")
except Exception as e:
    print(f"❌ Error (puede que el modelo no soporte tool calling): {e}")
    anotacion_tool = None

Ejecutando anotación con TOOL CALLING...

⏱️  Latencia: 5.1s | Método: fallback_texto

{
  "items_detectados": [
    5,
    12
  ],
  "escalas_afectadas": [
    "inhibicion",
    "control_emocional",
    "supervision_tarea"
  ],
  "nivel_alerta": "moderado",
  "nota_clinica": "Paciente muestra mejoría en la regulación emocional y eficiencia en tareas escolares tras inicio de medicación. Persiste hiperactividad motora en el entorno escolar.",
  "justificacion": "Se observa una reducción del tiempo en deberes (mejora en supervisión de tarea) y mayor tranquilidad en casa (control emocional), sin embargo, persiste la conducta de levantarse de la silla (déficit en inhibición)."
}


### 8.3 Comparar los dos mecanismos sobre el mismo texto

Anotamos la misma observación con los dos enfoques y comparamos. Las preguntas que
respondemos son las de la tesis:

1. **¿Cuál cumple el formato sin necesidad de recuperación?** (robustez estructural)
2. **¿Coinciden en los ítems detectados?** (concordancia entre métodos)
3. **¿Cuál pasa más métricas de evaluación técnica?** (calidad de salida)

In [16]:
# Anotación con el método clásico (prompt + parse), reutilizando Fases 3-5
print("Ejecutando anotación con PROMPT + PARSE JSON...\n")
t0 = time.time()
cruda_prompt = llamar_modelo(SYSTEM_PROMPT, construir_prompt_usuario(mensaje))
lat_prompt = time.time() - t0
anotacion_prompt = parsear_json(cruda_prompt)

# Tabla comparativa
def resumen_metodo(anotacion, metodo, latencia):
    if not anotacion:
        return {"metodo": metodo, "formato_ok": False, "n_items": 0,
                "nivel": "—", "latencia": latencia, "eval": None}
    ev = evaluar_anotacion(anotacion, instrumento)
    n_pasa = sum(1 for r in ev.values() if r["pasa"])
    return {
        "metodo": metodo,
        "formato_ok": True,
        "n_items": len(anotacion.get("items_detectados", [])),
        "items": sorted(anotacion.get("items_detectados", [])),
        "nivel": anotacion.get("nivel_alerta", "—"),
        "latencia": latencia,
        "eval": f"{n_pasa}/{len(ev)}",
    }

r_prompt = resumen_metodo(anotacion_prompt, "Prompt + parse", lat_prompt)
r_tool   = resumen_metodo(anotacion_tool,   "Tool calling",   lat_tool if anotacion_tool else 0)

print("COMPARACIÓN DE MECANISMOS\n" + "="*55)
for r in [r_prompt, r_tool]:
    print(f"\n{r['metodo']}:")
    print(f"  Formato válido:   {'✅' if r['formato_ok'] else '❌'}")
    print(f"  Ítems detectados: {r['n_items']} → {r.get('items', [])}")
    print(f"  Nivel alerta:     {r['nivel']}")
    print(f"  Latencia:         {r['latencia']:.1f}s")
    print(f"  Eval técnica:     {r['eval']}")

# Concordancia entre métodos
if anotacion_prompt and anotacion_tool:
    s1 = set(anotacion_prompt.get("items_detectados", []))
    s2 = set(anotacion_tool.get("items_detectados", []))
    comunes = s1 & s2
    union = s1 | s2
    jaccard = len(comunes)/len(union) if union else 0
    print(f"\n{'='*55}")
    print(f"CONCORDANCIA entre métodos (índice de Jaccard): {jaccard:.2f}")
    print(f"  Ítems en común:        {sorted(comunes)}")
    print(f"  Solo prompt+parse:     {sorted(s1 - s2)}")
    print(f"  Solo tool calling:     {sorted(s2 - s1)}")

Ejecutando anotación con PROMPT + PARSE JSON...

COMPARACIÓN DE MECANISMOS

Prompt + parse:
  Formato válido:   ✅
  Ítems detectados: 1 → [30]
  Nivel alerta:     moderado
  Latencia:         6.6s
  Eval técnica:     4/5

Tool calling:
  Formato válido:   ✅
  Ítems detectados: 2 → [5, 12]
  Nivel alerta:     moderado
  Latencia:         5.1s
  Eval técnica:     4/5

CONCORDANCIA entre métodos (índice de Jaccard): 0.00
  Ítems en común:        []
  Solo prompt+parse:     [30]
  Solo tool calling:     [5, 12]
